# Neural Translation Model

The goal is create a neural network that translates from English to German. For this I will use concepts like building more flexible model architectures, freezing layers, data processing pipeline and sequence modelling. 

### Some imports

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [37]:
# Importing libraries
import tensorflow as tf
import tensorflow_hub as hub
import unicodedata
import re

##
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Layer, Input, Masking, LSTM, Embedding, Dense
from tensorflow.keras.models import Model

![Flags overview image](data/germany_uk_flags.png)

For this project, I will use a language dataset from http://www.manythings.org/anki/ to build a neural translation model. This dataset consists of over 200,000 pairs of sentences in English and German. I will restrict the dataset to 60,000 pairs.


So, the goal is to develop a neural translation model from English to German, making use of a pre-trained English word embedding module.

In [3]:
# Loading the dataset

NUM_EXAMPLES = 40000
data_examples = []

with open('data/deu.txt', 'r', encoding='utf-8') as file:
    for line in file.readlines():
        if len(data_examples) < NUM_EXAMPLES:
            data_examples.append(line)
        else:
            break

In [4]:
#We can see that our dataset have 40000 examples right now.

print(len(data_examples))

#Looking some examples of the restricted dataset
for i in range(5):
    print(data_examples[i])

40000
Hi.	Hallo!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #380701 (cburgmer)

Hi.	Grüß Gott!	CC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #659813 (Esperantostern)

Run!	Lauf!	CC-BY 2.0 (France) Attribution: tatoeba.org #906328 (papabear) & #941078 (Fingerhut)

Wow!	Potzdonner!	CC-BY 2.0 (France) Attribution: tatoeba.org #52027 (Zifre) & #2122382 (Pfirsichbaeumchen)

Wow!	Donnerwetter!	CC-BY 2.0 (France) Attribution: tatoeba.org #52027 (Zifre) & #2122391 (Pfirsichbaeumchen)



In [5]:

#Functions to preprocess English and German sentences
def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = sentence.lower().strip()
    sentence = re.sub(r"ü", 'ue', sentence)
    sentence = re.sub(r"ä", 'ae', sentence)
    sentence = re.sub(r"ö", 'oe', sentence)
    sentence = re.sub(r'ß', 'ss', sentence)
    
    sentence = unicode_to_ascii(sentence)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"[^a-z?.!,']+", " ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)

    return sentence.strip()


## The custom translation model

The next image is a schematic of the custom translation model architecture that I will develop in this project.
![Model Schematic](data/neural_translation_model.png)

Key:
![Model key](data/neural_translation_model_key.png)


The custom model consists of an encoder RNN and a decoder RNN. The encoder takes words of an English sentence as input, and uses a pre-trained word embedding to embed the words into a 128-dimensional space. To indicate the end of the input sentence, a special end token (in the same 128-dimensional space) is passed in as an input. This token is a TensorFlow Variable that is learned in the training phase (unlike the pre-trained word embedding, which is frozen).

The decoder RNN takes the internal state of the encoder network as its initial state. A start token is passed in as the first input, which is embedded using a learned German word embedding. The decoder RNN then makes a prediction for the next German word, which during inference is then passed in as the following input, and this process is repeated until the special `<end>` token is emitted from the decoder.




# 1. Text Preprocessing

### Creating a separate list of English and German sentences, and preprocess them using the preprocess_sentence function.

In [6]:
#We can see that our dataset have 40000 examples right now.
print(len(data_examples))

#Looking some examples of the restricted dataset
for i in range(10,15):
    print(data_examples[i])

40000
Go on.	Mach weiter.	CC-BY 2.0 (France) Attribution: tatoeba.org #2230774 (CK) & #6625701 (Felixjp)

Hello!	Hallo!	CC-BY 2.0 (France) Attribution: tatoeba.org #373330 (CK) & #380701 (cburgmer)

I ran.	Ich rannte.	CC-BY 2.0 (France) Attribution: tatoeba.org #5828607 (CK) & #6643977 (Felixjp)

I see.	Ich verstehe.	CC-BY 2.0 (France) Attribution: tatoeba.org #2111796 (CK) & #547386 (Espi)

I see.	Aha.	CC-BY 2.0 (France) Attribution: tatoeba.org #2111796 (CK) & #5677459 (eric2)



In [7]:
english_sentences = []
german_sentences = []

for sentence in data_examples:
    parts = sentence.split('\t')
    english_sentences.append(parts[0])
    german_sentences.append(parts[1])

print("======= Looking the list of English Sentences: =======")
for i in range(5):
    print(f"{i}: {english_sentences[i]}")

print("======= Looking the list of German Sentences: =======")

for i in range(5):
    print(f"{i}: {german_sentences[i]}")

# Preprocessing the sentences:

preproc_english_sentences = [preprocess_sentence(s) for s in english_sentences]

print("======= Looking the list of Preprocess English Sentences: =======")
for i in range(5):
    print(f"{i}: {preproc_english_sentences[i]}")


preproc_german_sentences = [preprocess_sentence(s) for s in german_sentences]

print("======= Looking the list of Preprocess German Sentences: =======")

for i in range(5):
    print(f"{i}: {preproc_german_sentences[i]}")

#Checking the len of the preprocess lists

print(f"Longitud of preprocess_english_sentences: {len(preproc_english_sentences)}")
print(f"Longitud of preprocess_german_sentences: {len(preproc_english_sentences)}")




======= Looking the list of English Sentences: =======
0: Hi.
1: Hi.
2: Run!
3: Wow!
4: Wow!
======= Looking the list of German Sentences: =======
0: Hallo!
1: Grüß Gott!
2: Lauf!
3: Potzdonner!
4: Donnerwetter!
======= Looking the list of Preprocess English Sentences: =======
0: hi .
1: hi .
2: run !
3: wow !
4: wow !
======= Looking the list of Preprocess German Sentences: =======
0: hallo !
1: gruess gott !
2: lauf !
3: potzdonner !
4: donnerwetter !
Longitud of preprocess_english_sentences: 40000
Longitud of preprocess_german_sentences: 40000


### Adding a special `"<start>"` and `"<end>"` token to the beginning and end of every German sentence.

In [8]:

german_sentence_with_tokens = ["<start> " + s + " <end>" for s in preproc_german_sentences]
print("======= Looking the list of German Sentences Preprocessed with tokens: =======")
for i in range(5):
    print(f"{i}: {german_sentence_with_tokens[i]}")


======= Looking the list of German Sentences Preprocessed with tokens: =======
0: <start> hallo ! <end>
1: <start> gruess gott ! <end>
2: <start> lauf ! <end>
3: <start> potzdonner ! <end>
4: <start> donnerwetter ! <end>


### Using the Tokenizer class to tokenize the German Sentences

In [9]:

def create_sentence_tokenizer(list_of_sentences):
    tokenizer = Tokenizer(
        num_words=None,
        filters ="",
        lower=False,
        char_level=False,
        oov_token='<UNK>'
    )
    tokenizer.fit_on_texts(list_of_sentences)
    return tokenizer

In [10]:

german_tokenizer = create_sentence_tokenizer(german_sentence_with_tokens)
tokenizer_config = german_tokenizer.get_config()
print(tokenizer_config)
print(tokenizer_config.keys())
print(tokenizer_config['word_counts'])

{'num_words': None, 'filters': '', 'lower': False, 'split': ' ', 'char_level': False, 'oov_token': '<UNK>', 'document_count': 40000, 'word_counts': '{"<start>": 40000, "hallo": 32, "!": 3145, "<end>": 40000, "gruess": 3, "gott": 19, "lauf": 8, "potzdonner": 2, "donnerwetter": 1, "feuer": 19, "hilfe": 87, "zu": 1606, "huelf": 1, "stopp": 2, "warte": 37, "mach": 181, "weiter": 108, ".": 31336, "ich": 9061, "rannte": 39, "verstehe": 37, "aha": 1, "probiere": 6, "es": 3451, "hab": 42, "gewonnen": 62, "habe": 1638, "laecheln": 18, "zum": 128, "wohl": 39, "iss": 19, "auf": 900, "keine": 287, "bewegung": 4, "stehenbleiben": 2, "kapiert": 2, "?": 5584, "verstanden": 18, "einverstanden": 31, "er": 2051, "lief": 20, "mit": 490, "drueck": 5, "mich": 1279, "nimm": 79, "in": 661, "den": 532, "arm": 45, "umarme": 2, "fiel": 27, "hin": 88, "stuerzte": 6, "bin": 1551, "hingefallen": 1, "gestuerzt": 1, "weiss": 288, "gelogen": 23, "verloren": 121, "bezahlt": 28, "zahlte": 3, "sang": 18, "schwimme": 7, 

### Printting out 10 randomly chosen examples of (preprocessed) English and German sentence pairs. 

In [11]:
indices = np.random.choice(len(preproc_english_sentences), 5, replace=False)
german_sequences = german_tokenizer.texts_to_sequences(german_sentence_with_tokens)

for i in indices:
    print(f"English: {preproc_english_sentences[i]}")
    print(f"German (texto): {german_sentence_with_tokens[i]}")
    print(f"German (secuencia): {german_sequences[i]}")
    print()

English: did tom kiss you ?
German (texto): <start> hat tom sie gekuesst ? <end>
German (secuencia): [2, 15, 5, 9, 616, 8, 3]

English: we're speechless .
German (texto): <start> wir sind sprachlos . <end>
German (secuencia): [2, 17, 27, 2362, 4, 3]

English: are you injured ?
German (texto): <start> seid ihr verletzt ? <end>
German (secuencia): [2, 85, 28, 366, 8, 3]

English: we'll continue .
German (texto): <start> wir werden fortfahren . <end>
German (secuencia): [2, 17, 80, 2622, 4, 3]

English: she looked lonely .
German (texto): <start> sie hat einsam ausgesehen . <end>
German (secuencia): [2, 9, 15, 635, 4581, 4, 3]



### Padding the end of the tokenized German sequence with zeros, and batch the complete set of sequences into one numpy array.

In [12]:

german_sequences = german_tokenizer.texts_to_sequences(german_sentence_with_tokens)

german_padded = pad_sequences(german_sequences, padding='post')

print(type(german_padded))  
print(german_padded.shape)  

<class 'numpy.ndarray'>
(40000, 19)


# 2. Preparing the data with tf.data.Dataset object

### Loading the embedding layer

use a pre-trained English word embedding module from TensorFlow Hub. The URL for the module is https://tfhub.dev/google/tf2-preview/nnlm-en-dim128-with-normalization/1. This module has also been made available as a complete saved model in the folder `'./models/tf2-preview_nnlm-en-dim128_1'`.

This embedding takes a batch of text tokens in a 1-D tensor of strings as input. It then embeds the separate tokens into a 128-dimensional space. 

In [13]:
loaded_module = tf.saved_model.load("models/tf2-preview_nnlm-en-dim128_1")

class HubEmbeddingLayer(keras.layers.Layer):
    def __init__(self, module, **kwargs):
        super().__init__(**kwargs)
        self.module = module

    def call(self, inputs):
        return self.module(inputs)

embedding_layer = HubEmbeddingLayer(loaded_module)

In [14]:
#Testing the layer
embedding_layer(tf.constant(["these", "aren't", "the", "droids", "you're", "looking", "for"])).shape


TensorShape([7, 128])

### Creating a random training and validation set split

In [15]:

train_english, val_english, train_german, val_german = train_test_split(
    preproc_english_sentences, german_padded, test_size=0.2,
    random_state=42
)

print(len(train_english), len(val_english))
print(train_german.shape, val_german.shape)

32000 8000
(32000, 19) (8000, 19)


### Loading the training and validation sets into a td.data.Dataset object

In [16]:

train_dataset = tf.data.Dataset.from_tensor_slices((train_english, train_german))
val_dataset = tf.data.Dataset.from_tensor_slices((val_english, val_german))


In [17]:
for english, german in train_dataset.take(1):
    print(english)
    print(german)

tf.Tensor(b'how do they look ?', shape=(), dtype=string)
tf.Tensor(
[  2  40 105   9  41   8   3   0   0   0   0   0   0   0   0   0   0   0
   0], shape=(19,), dtype=int32)


### Creating a function to map over the datasets that splits eachs english sentence at spaces

In [18]:


def split_english(english, german):
    english_split = tf.strings.split(english)
    return english_split, german

train_dataset = train_dataset.map(split_english)
val_dataset = val_dataset.map(split_english)


In [19]:

for english, german in train_dataset.take(1):
    print(english)
    print(german)


tf.Tensor([b'how' b'do' b'they' b'look' b'?'], shape=(5,), dtype=string)
tf.Tensor(
[  2  40 105   9  41   8   3   0   0   0   0   0   0   0   0   0   0   0
   0], shape=(19,), dtype=int32)


### Creating a functon to map over the datasets that embeds each sequence of English words using the loaded embedding layer/model.

In [20]:

def embeds_each_sequence(english, german):
    english_embedded = embedding_layer(english)
    return english_embedded, german

train_dataset = train_dataset.map(embeds_each_sequence)
val_dataset = val_dataset.map(embeds_each_sequence)


In [21]:
for english, german in train_dataset.take(1):
    print(english)
    print(german)


tf.Tensor(
[[ 0.2070691  -0.03229755 -0.00846588 -0.1046224   0.10831352 -0.06459088
  -0.03458383 -0.14279097 -0.09368279 -0.08054009  0.09398187 -0.00136692
   0.01216753 -0.05182625  0.09590483 -0.01021192  0.16210379 -0.17628801
  -0.08603517  0.0709158   0.08362778  0.04175229  0.03637831  0.03587493
  -0.22489491  0.0655492  -0.00926939  0.06361149 -0.17090034 -0.04509904
   0.06768278  0.10349137 -0.11574525  0.10314068  0.04697882 -0.06267845
   0.01197376  0.05177886  0.04485051  0.01591656 -0.03640359 -0.01844085
   0.12275048  0.06692982 -0.08304647 -0.03745353  0.11176979  0.04442611
  -0.04106882 -0.05751404 -0.01733509  0.18284039  0.05168724 -0.02661607
   0.02426871 -0.15024377 -0.01166414 -0.1302475  -0.09341951  0.02161594
  -0.01557641  0.1911778  -0.04256317 -0.17760964 -0.04819937 -0.02984803
  -0.09816794 -0.03179838  0.05102905  0.1036388  -0.01119657  0.07379814
  -0.01864515 -0.05159878 -0.08748318  0.14904007  0.06235936 -0.14571123
  -0.12124559 -0.09996664  

### Creating a function to filter out dataset examples where the English setence is more than 13 (embedded) tokens in length.

In [22]:

def filter_long_sentences(english, german):
    return tf.shape(english)[0] <= 13

train_dataset = train_dataset.filter(filter_long_sentences)
val_dataset = val_dataset.filter(filter_long_sentences)


In [23]:
count_after_filter = sum(1 for s in train_english if len(s.split()) <= 13)
print(count_after_filter)

32000


### Creating a function to map over the datasets that pads each English sequence of embeddings with some distinct padding value before the sequece. 

In [24]:


def pad_english(english, german):
    current_length = tf.shape(english)[0]
    pad_amount = 13 - current_length
    
    english_padded = tf.pad(
        english,
        paddings=[[pad_amount, 0], [0, 0]],
        constant_values=-1
    )
    english_padded = tf.ensure_shape(english_padded, [13, 128])
    return english_padded, german

train_dataset = train_dataset.map(pad_english)
val_dataset = val_dataset.map(pad_english)


In [25]:
for english, german in train_dataset.take(1):
    print(english.shape)
    print(german.shape)

(13, 128)
(19,)


### Batching both training and validation Datasets with a batch of size of 16

In [26]:

train_dataset = train_dataset.batch(16)
val_dataset = val_dataset.batch(16)

### Printting the element_spec property

In [27]:
print(train_dataset.element_spec)
print(val_dataset.element_spec)

(TensorSpec(shape=(None, 13, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 19), dtype=tf.int32, name=None))
(TensorSpec(shape=(None, 13, 128), dtype=tf.float32, name=None), TensorSpec(shape=(None, 19), dtype=tf.int32, name=None))


### Printting the shape of the English Data Example from the training Dataset

In [28]:

for english, german in train_dataset.take(1):
    print(english.shape)


(16, 13, 128)


### Printing the German data exanmple Tensor form the validation Dataset

In [29]:

for english, german in val_dataset.take(1):
    print(german.shape)

(16, 19)


# 3. Creating the Custom Layer

The idea is create a custom layer to add the learned end token embedding to the encoder model:

![Encoder schematic](data/neural_translation_model_encoder.png)

### Creating a custom layer that adds a learned, 128-dimensional embedded 'end' token to each sequence in the batch

In [30]:
class AddEndToken(Layer):
    def build(self, input_shape):
        self.end_token = self.add_weight(
            shape=(128,),
            initializer="random_normal",
            trainable=True,
            name="end_token"
        )
    
    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]

        end_token_expanded = tf.reshape(self.end_token, [1,1,128])
        end_token_batch = tf.tile(end_token_expanded, [batch_size, 1, 1])

        return tf.concat([inputs, end_token_batch], axis=1)

### Testing the custom layer on a batch from the training Dataset

In [31]:


for english_batch, german_batch in train_dataset.take(1):
    print("Shape before:", english_batch.shape)

    add_end_token_layer = AddEndToken()
    output = add_end_token_layer(english_batch)
    print("Shape after:", output.shape)


Shape before: (16, 13, 128)
Shape after: (16, 14, 128)


# 4. Building the Encoder Network

### Building the encoder network using the Functional API

In [32]:

def build_encoder_network():
    encoder_inputs = Input(shape=(13,128))

    x = AddEndToken()(encoder_inputs)
    x = Masking(mask_value=-1)(x)

    lstm_output, hidden_state, cell_state = LSTM(512, return_state=True)(x)

    encoder_model = Model(inputs=encoder_inputs, outputs=[hidden_state, cell_state])

    return encoder_model

encoder_model = build_encoder_network()

### Testing the encoder model on a batch of English data from the training Dataset

In [33]:

for english_batch, german_batch in train_dataset.take(1):
    hidden_state, cell_state = encoder_model(english_batch)
    print("Hidden state shape:", hidden_state)
    print("Cell state shape:", cell_state)

Hidden state shape: tf.Tensor(
[[ 9.3130786e-03  9.1699872e-04  7.5785345e-03 ... -3.6613259e-03
  -1.4237896e-02  4.6081813e-03]
 [ 1.0844199e-02  2.6259306e-03  3.9111935e-03 ... -3.9290721e-03
  -1.4435228e-04  9.1950940e-03]
 [ 6.3953437e-03  8.4566632e-03  7.6598946e-05 ... -1.2025339e-02
  -4.1261669e-03  9.1341026e-03]
 ...
 [ 7.8765750e-03 -4.6222005e-03 -1.8276375e-04 ... -1.2015365e-02
  -3.8804172e-03  5.9749796e-03]
 [ 1.5681475e-02  9.2949942e-03  9.4239064e-04 ... -1.0258358e-02
  -1.0790923e-02  5.9977951e-03]
 [ 4.4005774e-03  1.1610194e-02  1.3084358e-02 ... -2.6768777e-03
  -1.4347167e-02  3.4801550e-03]], shape=(16, 512), dtype=float32)
Cell state shape: tf.Tensor(
[[ 0.01906819  0.00181556  0.01527572 ... -0.00729618 -0.02885959
   0.00924013]
 [ 0.02211424  0.00518386  0.00788737 ... -0.00787    -0.00029281
   0.01836178]
 [ 0.01303768  0.01672939  0.00015399 ... -0.02405741 -0.00838989
   0.01826636]
 ...
 [ 0.01610258 -0.00911532 -0.00036746 ... -0.02406365 -0.00

### Printing the encoder model summary

In [36]:
print(len(encoder_model.trainable_variables))  

4


In [35]:
encoder_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 13, 128)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_end_token_1     │ (None, 14, 128)   │        128 │ input_layer[0][0] │
│ (AddEndToken)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 14, 128)   │          0 │ add_end_token_1[… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking (Masking)   │ (None, 14, 128)   │          0 │ add_end_token_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, 14)        │          0 │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 512),     │  1,312,768 │ masking[0][0],    │
│                     │ (None, 512),      │            │ any[0][0]         │
│                     │ (None, 512)]      │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,312,896 (5.01 MB)

 Trainable params: 1,312,896 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

# 5. Building the Decoder Network

The decoder network follows the schematic diagram below. 

![Decoder schematic](data/neural_translation_model_decoder.png)

### Building the decoder network using Model subclassing

In [43]:

class Decoder(Model):
    def __init__(self, vocab_size, **kwargs):
        super().__init__(**kwargs)

        self.embeddings = Embedding(
            input_dim=vocab_size,
            output_dim=128,
            mask_zero=True
        )

        self.lstm = LSTM(
            units=512,
            return_state = True,
            return_sequences = True
        )

        self.dense = Dense(units=vocab_size)


    def call(self, inputs, hidden_state=None, cell_state=None):

        x = self.embeddings(inputs)

        if hidden_state is not None and cell_state is not None:
            x, state_h, state_c = self.lstm(x, initial_state=[hidden_state, cell_state])
        else:
            x, state_h, state_c = self.lstm(x)

        output = self.dense(x)

        return output, state_h, state_c


vocab_size = len(german_tokenizer.word_index) + 1
print(vocab_size)

decoder_model = Decoder(vocab_size)


9071


### Testing the decoder model on a batch of English and German data from the training Dataset

In [46]:
for english_batch, german_batch in train_dataset.take(1):
    hidden_state, cell_state = encoder_model(english_batch)

    decoder_output, decoder_hidden, decoder_cell = decoder_model(
        german_batch,
        hidden_state=hidden_state,
        cell_state=cell_state)

    print(decoder_output.shape)


(16, 19, 9071)


In [48]:
print(vocab_size)
print(len(german_tokenizer.word_index))

9071
9070


### Printing the decoder model summary

In [47]:
decoder_model.summary()


Model: "decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (16, 19, 128)          │     1,161,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ((16, 19, 512), (16,   │     1,312,768 │
│                                 │ 512), (16, 512))       │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (16, 19, 9071)         │     4,653,423 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,127,279 (27.19 MB)

 Trainable params: 7,127,279 (27.19 MB)

 Non-trainable params: 0 (0.00 B)

# 6. Make a custom training loop